In [1]:
import sqlite3
import pandas as pd

In [2]:
# Load cleaned data
df = pd.read_excel("../data/Retail_IQ_Sales_Cleaned.xlsx")

In [3]:
# Normalize column names: lowercase, remove spaces, replace with underscores
df.columns = (
    df.columns
      .str.strip()        # remove leading/trailing spaces
      .str.lower()        # make all lowercase
      .str.replace(' ', '_')  # replace spaces with underscore
)

In [4]:
# Connect to SQLite database
conn = sqlite3.connect("../database/retailiq.db")
cursor = conn.cursor()

In [5]:
# 🔹 Preview table columns to make sure names are correct
print("Columns in sales_data table:")
cursor.execute("PRAGMA table_info(sales_data);")
columns = cursor.fetchall()
for col in columns:
    print(col)

Columns in sales_data table:
(0, 'Item Fat Content', 'TEXT', 0, None, 0)
(1, 'Item Identifier', 'TEXT', 0, None, 0)
(2, 'Item Type', 'TEXT', 0, None, 0)
(3, 'Outlet Establishment Year', 'INTEGER', 0, None, 0)
(4, 'Outlet Identifier', 'TEXT', 0, None, 0)
(5, 'Outlet Location Type', 'TEXT', 0, None, 0)
(6, 'Outlet Size', 'TEXT', 0, None, 0)
(7, 'Outlet Type', 'TEXT', 0, None, 0)
(8, 'Item Visibility', 'REAL', 0, None, 0)
(9, 'Item Weight', 'REAL', 0, None, 0)
(10, 'Sales', 'REAL', 0, None, 0)
(11, 'Rating', 'REAL', 0, None, 0)


In [6]:
# 🔹 Preview first few rows
print("\nFirst 5 rows of sales_data:")
df_preview = pd.read_sql("SELECT * FROM sales_data LIMIT 5", conn)
print(df_preview)


First 5 rows of sales_data:
  Item Fat Content Item Identifier              Item Type  \
0          Regular           FDX32  Fruits And Vegetables   
1          Low Fat           NCB42     Health And Hygiene   
2          Regular           FDR28           Frozen Foods   
3          Regular           FDL50                 Canned   
4          Low Fat           DRI25            Soft Drinks   

   Outlet Establishment Year Outlet Identifier Outlet Location Type  \
0                       2012            OUT049               TIER 1   
1                       2022            OUT018               TIER 3   
2                       2016            OUT046               TIER 1   
3                       2014            OUT013               TIER 3   
4                       2015            OUT045               TIER 2   

  Outlet Size        Outlet Type  Item Visibility  Item Weight     Sales  \
0      Medium  Supermarket Type1         0.100014        15.10  145.4786   
1      Medium  Supermarke

In [7]:
# 1️⃣ Total Sales
total_sales = pd.read_sql("""
SELECT SUM(sales) AS total_sales
FROM sales_data
""", conn)
print("\nTotal Sales:")
print(total_sales)


Total Sales:
   total_sales
0  997159.2358


In [10]:
# 2️⃣ Sales by Item Type
sales_by_item = pd.read_sql("""
SELECT item_type, SUM(sales) AS total_sales
FROM sales_data
GROUP BY item_type
ORDER BY total_sales DESC
""", conn)
print("\nSales by Item Type:")
print(sales_by_item)

DatabaseError: Execution failed on sql '
SELECT item_type, SUM(sales) AS total_sales
FROM sales_data
GROUP BY item_type
ORDER BY total_sales DESC
': no such column: item_type

In [41]:
# 3️⃣ Sales by Outlet Location Type
sales_by_location = pd.read_sql("""
SELECT outlet_location_type, SUM(sales) AS total_sales
FROM sales_data
GROUP BY outlet_location_type
ORDER BY total_sales DESC
""", conn)
print("\nSales by Outlet Location Type:")
print(sales_by_location)


Sales by Outlet Location Type:
  outlet_location_type  total_sales
0               TIER 2  393150.6476
1               TIER 3  341418.3586
2               TIER 1  262590.2296


In [43]:
# 4️⃣ Monthly/Yearly Sales Trend
# Note: We only have year info via 'outlet_establishment_year', no month, so using that for trend
monthly_sales = pd.read_sql("""
SELECT outlet_establishment_year AS year, SUM(sales) AS total_sales
FROM sales_data
GROUP BY outlet_establishment_year
ORDER BY outlet_establishment_year
""", conn)
print("\nYearly Sales Trend:")
print(monthly_sales)


Yearly Sales Trend:
   year  total_sales
0  2011   78131.5666
1  2012  130476.8598
2  2014  131809.0156
3  2015  130942.7802
4  2016  132113.3698
5  2017  133103.9070
6  2020  129103.9604
7  2022  131477.7764


In [44]:
# 5️⃣ Average Rating per Item Type
avg_rating = pd.read_sql("""
SELECT item_type, ROUND(AVG(rating),2) AS avg_rating
FROM sales_data
GROUP BY item_type
""", conn)
print("\nAverage Rating by Item Type:")
print(avg_rating)


Average Rating by Item Type:
                item_type  avg_rating
0            Baking Goods        3.98
1                  Breads        3.89
2               Breakfast        3.94
3                  Canned        4.00
4                   Dairy        3.97
5            Frozen Foods        3.97
6   Fruits And Vegetables        3.95
7             Hard Drinks        3.95
8      Health And Hygiene        3.98
9               Household        3.99
10                   Meat        4.03
11                 Others        3.91
12                Seafood        3.93
13            Snack Foods        3.96
14            Soft Drinks        3.90
15          Starchy Foods        3.88


In [45]:
conn.commit()